# JPEG AI as a Threat to Deepfake Detection
**Computer Vision — Prof. Irene Amerini — Spring 2026**

Project structure:
1. **Imports** — all required packages
2. **Globals** — project-wide constants and paths
3. **Utils** — helper functions
4. **Data** — dataset loading and JPEG AI compression pipeline
5. **Network** — deepfake detector definitions
6. **Train** — fine-tuning loop (mitigation strategy 1)
7. **Evaluation** — metrics, frequency analysis, mitigation comparison

---
## 1. Imports

In [ ]:
import os
import sys
import random
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
import cv2
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, roc_curve

# Project modules
sys.path.insert(0, str(Path.cwd()))
from src.compression.jpegai_codec import compress_dataset, BPP_LEVELS

print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()} | MPS: {torch.backends.mps.is_available()}")

---
## 2. Globals

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Device ───────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")   # Apple Silicon
else:
    DEVICE = torch.device("cpu")
print(f"Using device: {DEVICE}")

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT          = Path.cwd()
DATA_DIR      = ROOT / "data"
ORIGINAL_DIR  = DATA_DIR / "original"     # raw dataset images
COMPRESSED_DIR= DATA_DIR / "compressed"   # JPEG AI outputs
RESULTS_DIR   = ROOT / "results"
CHECKPOINTS_DIR = ROOT / "checkpoints"

for d in [DATA_DIR, ORIGINAL_DIR, COMPRESSED_DIR, RESULTS_DIR, CHECKPOINTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Compression settings ──────────────────────────────────────────────────────
# BPP levels for the degradation curve
BPP_LEVELS    = [0.1, 0.3, 0.5, 0.8, 1.0, 2.0]
JPEGAI_PROFILE= "base"

# ── Training hyperparameters ──────────────────────────────────────────────────
BATCH_SIZE    = 32
NUM_EPOCHS    = 10
LR            = 1e-4
IMG_SIZE      = 224

# ── Dataset split ─────────────────────────────────────────────────────────────
# Expected folder structure inside ORIGINAL_DIR:
#   original/
#     real/   ← genuine images   (label 0)
#     fake/   ← deepfake images  (label 1)
LABEL_MAP = {"real": 0, "fake": 1}

---
## 3. Utils

In [ ]:
def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def compute_power_spectrum(img_gray: np.ndarray) -> np.ndarray:
    """2D power spectrum (log magnitude) of a grayscale image."""
    f = np.fft.fft2(img_gray.astype(np.float32))
    fshift = np.fft.fftshift(f)
    return 20 * np.log(np.abs(fshift) + 1e-8)


def azimuthal_average(spectrum_2d: np.ndarray) -> np.ndarray:
    """Radial (azimuthal) average of a 2D spectrum → 1D frequency profile."""
    h, w = spectrum_2d.shape
    cy, cx = h // 2, w // 2
    y, x = np.ogrid[:h, :w]
    r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2).astype(int)
    max_r = min(cy, cx)
    profile = np.array([spectrum_2d[r == i].mean() for i in range(max_r)])
    return profile


def compute_dct_histogram(img_gray: np.ndarray, bins: int = 256) -> tuple:
    """DCT coefficient histogram of a grayscale image (block size 8×8)."""
    h, w = img_gray.shape
    h = (h // 8) * 8
    w = (w // 8) * 8
    img = img_gray[:h, :w].astype(np.float32)
    coeffs = []
    for i in range(0, h, 8):
        for j in range(0, w, 8):
            block = img[i:i+8, j:j+8]
            dct = cv2.dct(block)
            coeffs.append(dct.flatten())
    all_coeffs = np.concatenate(coeffs)
    hist, edges = np.histogram(all_coeffs, bins=bins, range=(-100, 100))
    return hist, edges


def evaluate_detector(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device = DEVICE,
) -> dict:
    """Run inference and return AUC, accuracy, F1."""
    model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for imgs, labels in dataloader:
            imgs = imgs.to(device)
            logits = model(imgs)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())

    preds = (np.array(all_probs) >= 0.5).astype(int)
    return {
        "auc":      roc_auc_score(all_labels, all_probs),
        "accuracy": accuracy_score(all_labels, preds),
        "f1":       f1_score(all_labels, preds),
        "labels":   np.array(all_labels),
        "probs":    np.array(all_probs),
    }


def plot_degradation_curve(results: dict, metric: str = "auc", title: str = "") -> None:
    """
    Plot detector metric vs BPP.
    results: {detector_name: {bpp: metric_value}}
    """
    fig, ax = plt.subplots(figsize=(8, 5))
    for detector_name, bpp_metrics in results.items():
        bpps = sorted(bpp_metrics.keys())
        vals = [bpp_metrics[b] for b in bpps]
        ax.plot(bpps, vals, marker="o", label=detector_name)
    ax.set_xlabel("BPP (bits per pixel)")
    ax.set_ylabel(metric.upper())
    ax.set_title(title or f"Detector {metric.upper()} vs JPEG AI BPP")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"degradation_{metric}.png", dpi=150)
    plt.show()

---
## 4. Data
### 4.1 Dataset class

### 4.0 FaceForensics++ — Download & Reorganize

Download instructions:
```bash
# 1. Get the download script
curl -O http://kaldir.vc.in.tum.de/faceforensics_download_v4.py

# 2. Download real (original) frames — 500 videos, uncompressed (c0)
python faceforensics_download_v4.py data/ff++ --server EU2 -d original   -c c0 -t images --num_videos 500

# 3. Download fake (Deepfakes) frames
python faceforensics_download_v4.py data/ff++ --server EU2 -d Deepfakes  -c c0 -t images --num_videos 500
```

Then run the reorganization cell below to create the `real/` / `fake/` structure expected by the notebook.

In [ ]:
import shutil
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
FF_ROOT   = DATA_DIR / "ff++"          # raw FF++ download location
OUT_REAL  = ORIGINAL_DIR / "real"
OUT_FAKE  = ORIGINAL_DIR / "fake"
OUT_REAL.mkdir(parents=True, exist_ok=True)
OUT_FAKE.mkdir(parents=True, exist_ok=True)

# ── How many frames to take per video (to keep dataset balanced & manageable) ─
MAX_FRAMES_PER_VIDEO = 10

# ── Copy real frames ──────────────────────────────────────────────────────────
real_src = FF_ROOT / "original_sequences" / "actors" / "c0" / "images"
real_count = 0
if real_src.exists():
    for video_dir in sorted(real_src.iterdir()):
        if not video_dir.is_dir():
            continue
        frames = sorted(video_dir.glob("*.png")) + sorted(video_dir.glob("*.jpg"))
        for f in frames[:MAX_FRAMES_PER_VIDEO]:
            dst = OUT_REAL / f"{video_dir.name}_{f.name}"
            if not dst.exists():
                shutil.copy2(f, dst)
            real_count += 1
    print(f"Real frames copied: {real_count}")
else:
    print(f"Real source not found: {real_src}\nRun the download commands first.")

# ── Copy fake frames ──────────────────────────────────────────────────────────
fake_src = FF_ROOT / "manipulated_sequences" / "Deepfakes" / "c0" / "images"
fake_count = 0
if fake_src.exists():
    for video_dir in sorted(fake_src.iterdir()):
        if not video_dir.is_dir():
            continue
        frames = sorted(video_dir.glob("*.png")) + sorted(video_dir.glob("*.jpg"))
        for f in frames[:MAX_FRAMES_PER_VIDEO]:
            dst = OUT_FAKE / f"{video_dir.name}_{f.name}"
            if not dst.exists():
                shutil.copy2(f, dst)
            fake_count += 1
    print(f"Fake frames copied: {fake_count}")
else:
    print(f"Fake source not found: {fake_src}\nRun the download commands first.")

print(f"\nFinal dataset → real: {len(list(OUT_REAL.iterdir()))} | fake: {len(list(OUT_FAKE.iterdir()))}")


In [ ]:
class DeepfakeDataset(Dataset):
    """
    Loads images from a directory with 'real/' and 'fake/' subdirectories.
    Works for both original and JPEG-AI-compressed splits.
    """
    def __init__(self, root: Path, transform=None, label_map: dict = LABEL_MAP):
        self.samples = []
        for cls_name, label in label_map.items():
            cls_dir = root / cls_name
            if not cls_dir.exists():
                continue
            for p in sorted(cls_dir.rglob("*.png")) + sorted(cls_dir.rglob("*.jpg")):
                self.samples.append((p, label))
        self.transform = transform or T.Compose([
            T.Resize((IMG_SIZE, IMG_SIZE)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label


def make_dataloader(root: Path, batch_size: int = BATCH_SIZE, shuffle: bool = False) -> DataLoader:
    ds = DeepfakeDataset(root)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=4, pin_memory=True)


# Verify dataset structure
print("Original dataset:")
for cls in ["real", "fake"]:
    cls_dir = ORIGINAL_DIR / cls
    count = len(list(cls_dir.rglob("*.png"))) + len(list(cls_dir.rglob("*.jpg"))) if cls_dir.exists() else 0
    print(f"  {cls}: {count} images")

### 4.2 JPEG AI Compression Pipeline

> **Run this step on Linux (native) or via Docker (Mac).**  
> The compressed images are saved to `data/compressed/` and reused in all subsequent cells.  
> Skip this cell if `data/compressed/` already exists.

In [ ]:
# ── Option A: Python API (if running inside jpeg_ai_vm env) ──────────────────
# compress_dataset(
#     dataset_dir=ORIGINAL_DIR,
#     output_root=COMPRESSED_DIR,
#     bpp_levels=BPP_LEVELS,
#     profile=JPEGAI_PROFILE,
# )

# ── Option B: Shell script (Linux native or Docker) ──────────────────────────
# Run in terminal:
#   conda activate jpeg_ai_vm
#   bash scripts/compress_dataset.sh data/original data/compressed
#
# Or with Docker:
#   docker build -t jpegai-codec -f docker/Dockerfile .
#   docker run --rm -v $(pwd)/data:/workspace/project/data jpegai-codec \
#       bash scripts/compress_dataset.sh data/original data/compressed

# ── Verify compressed data exists ────────────────────────────────────────────
for bpp in BPP_LEVELS:
    bpp_tag = f"bpp_{int(bpp*100):03d}"
    bpp_dir = COMPRESSED_DIR / bpp_tag
    count = len(list(bpp_dir.rglob("*.png"))) if bpp_dir.exists() else 0
    status = "OK" if count > 0 else "MISSING"
    print(f"  [{status}] {bpp_tag}: {count} images")

---
## 5. Network
### 5.1 Detector definitions

In [ ]:
import timm

def load_detector(name: str, pretrained: bool = True, num_classes: int = 2) -> nn.Module:
    """
    Load a deepfake detector backbone.

    Supported names:
        'resnet50'        — CNNDetection-style (Wang et al., CVPR 2020)
        'efficientnet_b4' — EfficientNet (timm)
        'xception'        — FaceForensics++ baseline
    """
    model = timm.create_model(name, pretrained=pretrained, num_classes=num_classes)
    return model.to(DEVICE)


# TODO: load pre-trained forensics weights if available.
# Example: CNNDetection weights from https://github.com/peterwang512/CNNDetection
#
# detector = load_detector('resnet50', pretrained=False)
# ckpt = torch.load('checkpoints/cnndetection_resnet50.pth', map_location=DEVICE)
# detector.load_state_dict(ckpt)

detector_names = ["resnet50", "efficientnet_b4"]
print("Detectors:", detector_names)

---
## 6. Train
### 6.1 Fine-tuning loop (Mitigation Strategy 1: compression-augmented training)

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
    device: torch.device = DEVICE,
) -> float:
    model.train()
    total_loss = 0.0
    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)


def finetune(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = NUM_EPOCHS,
    lr: float = LR,
    save_path: Path = None,
) -> list:
    """Fine-tune detector. Returns list of val-AUC per epoch."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = []

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        metrics = evaluate_detector(model, val_loader)
        scheduler.step()
        history.append(metrics["auc"])
        print(f"Epoch {epoch:02d} | loss {train_loss:.4f} | val AUC {metrics['auc']:.4f}")

    if save_path:
        torch.save(model.state_dict(), save_path)
        print(f"Checkpoint saved: {save_path}")
    return history


# ── Compression-augmented dataset for fine-tuning ────────────────────────────
class AugmentedDeepfakeDataset(Dataset):
    """
    Mixes original + JPEG-AI-compressed images for robust fine-tuning.
    Randomly picks original or one compressed variant per sample each epoch.
    """
    def __init__(self, original_root: Path, compressed_root: Path,
                 bpp_levels: list = BPP_LEVELS, transform=None):
        self.base = DeepfakeDataset(original_root, transform)
        self.compressed_roots = [
            compressed_root / f"bpp_{int(b*100):03d}" for b in bpp_levels
        ]
        self.transform = self.base.transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        orig_path, label = self.base.samples[idx]
        # Randomly pick original or a compressed version
        candidates = [orig_path]
        for cr in self.compressed_roots:
            # Match compressed file by stem
            compressed_dir = cr / orig_path.parent.name
            matches = list(compressed_dir.glob(f"{orig_path.stem}_bpp*.png"))
            if matches:
                candidates.append(matches[0])
        chosen = random.choice(candidates)
        img = Image.open(chosen).convert("RGB")
        return self.transform(img), label


print("Train module ready.")

---
## 7. Evaluation
### 7.1 Phase 1 — Baseline degradation curve

In [ ]:
# Evaluate each detector on original + all BPP-compressed test sets.
# Results dict: {detector_name: {bpp_or_'original': {metric: value}}}

degradation_results = {}  # populated below

for det_name in detector_names:
    model = load_detector(det_name)
    # TODO: load forensics pre-trained weights here
    model.eval()

    degradation_results[det_name] = {}

    # Original (uncompressed)
    loader = make_dataloader(ORIGINAL_DIR)
    metrics = evaluate_detector(model, loader)
    degradation_results[det_name]["original"] = metrics
    print(f"{det_name} | original | AUC={metrics['auc']:.4f}")

    # Compressed at each BPP
    for bpp in BPP_LEVELS:
        bpp_tag = f"bpp_{int(bpp*100):03d}"
        bpp_dir = COMPRESSED_DIR / bpp_tag
        if not bpp_dir.exists():
            print(f"  SKIP {bpp_tag} (not found)")
            continue
        loader = make_dataloader(bpp_dir)
        metrics = evaluate_detector(model, loader)
        degradation_results[det_name][bpp] = metrics
        print(f"{det_name} | {bpp_tag} | AUC={metrics['auc']:.4f}")

# Plot
auc_by_bpp = {
    name: {k: v["auc"] for k, v in bpp_map.items() if k != "original"}
    for name, bpp_map in degradation_results.items()
}
plot_degradation_curve(auc_by_bpp, metric="auc")

### 7.2 Phase 2 — Frequency-domain forensic analysis

In [ ]:
# Compare power spectra across: original-real, original-fake, compressed-real, compressed-fake

N_SAMPLES = 100  # images per category

def collect_spectra(img_dir: Path, label: str, n: int = N_SAMPLES) -> list:
    paths = list(img_dir.rglob("*.png")) + list(img_dir.rglob("*.jpg"))
    paths = sorted(paths)[:n]
    spectra = []
    for p in paths:
        gray = np.array(Image.open(p).convert("L"))
        spectra.append(azimuthal_average(compute_power_spectrum(gray)))
    return spectra


fig, axes = plt.subplots(1, len(BPP_LEVELS) + 1, figsize=(20, 4), sharey=True)

# Original
for cls in ["real", "fake"]:
    specs = collect_spectra(ORIGINAL_DIR / cls, cls)
    mean_spec = np.mean(specs, axis=0)
    axes[0].plot(mean_spec, label=cls)
axes[0].set_title("Original")
axes[0].set_xlabel("Frequency (px⁻¹)")
axes[0].set_ylabel("Power (dB)")
axes[0].legend()

# Compressed at each BPP
for i, bpp in enumerate(BPP_LEVELS):
    bpp_tag = f"bpp_{int(bpp*100):03d}"
    ax = axes[i + 1]
    for cls in ["real", "fake"]:
        bpp_cls_dir = COMPRESSED_DIR / bpp_tag / cls
        if not bpp_cls_dir.exists():
            continue
        specs = collect_spectra(bpp_cls_dir, cls)
        mean_spec = np.mean(specs, axis=0)
        ax.plot(mean_spec, label=cls)
    ax.set_title(f"BPP={bpp}")
    ax.set_xlabel("Frequency (px⁻¹)")
    ax.legend()

plt.suptitle("Mean Azimuthal Power Spectra — JPEG AI Compression Effect", fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "power_spectra.png", dpi=150)
plt.show()

### 7.3 DCT coefficient distributions

In [ ]:
fig, axes = plt.subplots(2, len(BPP_LEVELS) + 1, figsize=(22, 7))

def plot_dct_hist(ax, img_dir: Path, label: str, color: str, n: int = 50) -> None:
    paths = (list(img_dir.rglob("*.png")) + list(img_dir.rglob("*.jpg")))[:n]
    all_hists = []
    for p in paths:
        gray = np.array(Image.open(p).convert("L"))
        hist, edges = compute_dct_histogram(gray)
        all_hists.append(hist / hist.sum())  # normalize
    centers = (edges[:-1] + edges[1:]) / 2
    mean_hist = np.mean(all_hists, axis=0)
    ax.plot(centers, mean_hist, color=color, label=label, linewidth=1.5)
    ax.set_xlim(-60, 60)
    ax.set_xlabel("DCT coefficient value")
    ax.set_ylabel("Normalized count")
    ax.legend(fontsize=8)

for row, cls in enumerate(["real", "fake"]):
    color = "steelblue" if cls == "real" else "tomato"
    # Original
    plot_dct_hist(axes[row][0], ORIGINAL_DIR / cls, f"original/{cls}", color)
    axes[row][0].set_title(f"Original – {cls}")
    # Compressed
    for i, bpp in enumerate(BPP_LEVELS):
        bpp_tag = f"bpp_{int(bpp*100):03d}"
        bpp_cls_dir = COMPRESSED_DIR / bpp_tag / cls
        ax = axes[row][i + 1]
        if bpp_cls_dir.exists():
            plot_dct_hist(ax, bpp_cls_dir, f"BPP={bpp}/{cls}", color)
        ax.set_title(f"BPP={bpp} – {cls}")

plt.suptitle("DCT Coefficient Distributions — Real vs Fake under JPEG AI", fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "dct_histograms.png", dpi=150)
plt.show()

### 7.4 Phase 3 — Mitigation comparison

In [ ]:
# ── Mitigation 1: Fine-tuning on compression-augmented data ──────────────────
# aug_dataset = AugmentedDeepfakeDataset(ORIGINAL_DIR / 'train', COMPRESSED_DIR)
# aug_loader  = DataLoader(aug_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
# val_loader  = make_dataloader(ORIGINAL_DIR / 'val')
#
# finetuned_model = load_detector('resnet50')
# # load pre-trained weights
# history = finetune(finetuned_model, aug_loader, val_loader,
#                    save_path=CHECKPOINTS_DIR / 'resnet50_finetuned.pth')

# ── Mitigation 2: Standard JPEG augmentation (cheap baseline) ────────────────
jpeg_aug_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomApply([T.Lambda(lambda img: Image.fromarray(
        cv2.imencode('.jpg', np.array(img), [cv2.IMWRITE_JPEG_QUALITY,
            random.randint(30, 95)])[1].tobytes()
        and cv2.imdecode(np.frombuffer(
            cv2.imencode('.jpg', np.array(img), [cv2.IMWRITE_JPEG_QUALITY,
            random.randint(30, 95)])[1], np.uint8), cv2.IMREAD_COLOR)
    ))], p=0.5),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ── Mitigation results summary table ─────────────────────────────────────────
# Populate after running both mitigations
mitigation_summary = pd.DataFrame(columns=["Detector", "Strategy", "BPP", "AUC", "Accuracy", "F1"])
# mitigation_summary.to_csv(RESULTS_DIR / 'mitigation_results.csv', index=False)
print("Mitigation cells ready. Run fine-tuning blocks above, then populate mitigation_summary.")

### 7.5 Final summary plot — before vs after mitigation

In [ ]:
# After all mitigations are evaluated, compare degradation curves:
#   - Baseline (no mitigation)
#   - Mitigation 1 (fine-tuning on augmented data)
#   - Mitigation 2 (JPEG augmentation)

# Example structure (populate with real results):
# compare = {
#     "Baseline":              {0.1: 0.61, 0.3: 0.68, 0.5: 0.74, 0.8: 0.80, 1.0: 0.83, 2.0: 0.89},
#     "Mitigation-1 (FT)": {0.1: 0.72, 0.3: 0.77, 0.5: 0.81, 0.8: 0.85, 1.0: 0.87, 2.0: 0.90},
#     "Mitigation-2 (JPEG aug)": {0.1: 0.65, 0.3: 0.71, 0.5: 0.76, 0.8: 0.81, 1.0: 0.84, 2.0: 0.89},
# }
# plot_degradation_curve(compare, metric="auc", title="Mitigation Comparison — ResNet50")

print("Populate 'compare' dict with real results and uncomment the plot call.")